# K-Mastery: OCR CRNN Training Notebook

This notebook trains a CRNN (Convolutional Recurrent Neural Network) on the synthetic OCR dataset to recognize Korean handwriting/signage.

In [ ]:
!pip install tensorflow keras matplotlib pillow

## 1. Upload Dataset
Upload the `ocr_dataset.zip` generated locally to Colab's workspace and unzip it.

In [ ]:

import zipfile
import os

# Assuming ocr_dataset.zip is uploaded to the root dir
if os.path.exists('/content/ocr_dataset.zip'):
    with zipfile.ZipFile('/content/ocr_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/dataset/')
    print("Dataset extracted!")
else:
    print("Please upload ocr_dataset.zip using the file browser on the left.")


## 2. Define CRNN Model Architecture

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model

def build_crnn_model(img_width=256, img_height=64, num_classes=1000):
    # Input shape
    input_img = layers.Input(shape=(img_width, img_height, 1), name="image", dtype="float32")
    
    # Convolutional layers
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="Conv1")(input_img)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)
    
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="Conv2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)
    
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="Conv3")(x)
    x = layers.MaxPooling2D((2, 1), name="pool3")(x)
    
    # Reshape for RNN
    new_shape = ((img_width // 8), (img_height // 4) * 128)
    x = layers.Reshape(target_shape=new_shape, name="reshape")(x)
    x = layers.Dense(64, activation="relu", name="dense1")(x)
    
    # RNNs
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.25))(x)
    
    # Output layer
    output = layers.Dense(num_classes + 1, activation="softmax", name="dense2")(x)
    
    model = Model(inputs=input_img, outputs=output, name="ocr_model_v1")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_crnn_model()
model.summary()


## 3. Train and Export
Run your training loop using the dataset, then export the weights to `.h5`.

In [ ]:

# model.fit(...)
# After training, save the weights:
model.save_weights('/content/ocr_weights.h5')
print("Model weights saved to ocr_weights.h5. Download this file and place it in the backend/models directory.")
